In [2]:
import numpy as np
import pandas as pd
pd.options.display.float_format = '{:,.3f}'.format
import csv
import sys
import jieba
from datetime import datetime
# import word2vec
import pyLDAvis
import pyLDAvis.sklearn
import matplotlib.pyplot as plt
import math
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_squared_log_error

plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['figure.figsize']=['600', '600']
plt.rcParams['axes.unicode_minus']=False

In [3]:
## Read documents
data = pd.read_csv("detail_topic_new_withStay.csv",header=None)
data.columns=["Auto_Index","Author","Title","Max_layer","Read","Time","Stay","M_ratio"]

red_pocket_title=[u'下雨天宅家里？万元巨款你不准备要了吗！<长沙转发有礼>',
u'【周年店庆邀请函】丨点开有惊喜，转发有红包',
u'名匠心，设计情丨2017名匠杯设计大赛盛大启动',
u'【转发有惊喜】名匠心，设计情丨2017名匠杯设计大赛盛大启动',
u'年度特权日 装修特权价｜名匠2017年度【消费者特权日】盛大启幕',
u'眼要急！手要快！装修特权卡全城抢疯了',
u'开业特权 尊享钜惠 | 名匠装饰集团5家地市新店齐开',
u'国庆装修大动作！如此疯狂钜惠怎能错过？（前3000名转发有红包）',
u'抢定年度特权卡，装修钜惠更疯狂！（转发有红包）',
u'抢定年度特权卡，装修钜惠更疯狂！']

In [4]:
## Cut words, clean

# Word cut function, by jieba
def word_cut(text):
    return " ".join(jieba.cut(text))

data['cutted']=data.Title.apply(word_cut)
cut_list=[] # the list of cutted

# Remove empty char
for line in data['cutted']:
    t=line.split(' ')
    while '' in t:
        t.remove('')
    cut_list.append(t)

# Read stop words
stop_words = open('stopwords.txt',encoding="utf-8")
stop_words_text = stop_words.read()
stop_words.close()
stop_words_list = stop_words_text.split('\n')

# Clean words in cut_list
cut_list_re = [] # the list of words after removal. [[1,2,..],[],..]
cut_list_re_str = [] # the str version. ["1 2 ..","",..]
word_list=[] # total words list
for line in cut_list:
    t=[]
    for word in line:
        if word not in stop_words_list and word!=" " and len(word)!=1 and not word.isdigit() and word.isalpha():
            t.append(word)
            if word not in word_list: word_list.append(word)
    cut_list_re.append(t)
    cut_list_re_str.append(" ".join(t))

data.cutted=cut_list_re_str # cover previous cutted by words after removal
# remove meaning-less data
# used in topic model, the titles don't contain such materials
data=data[ ~ data['cutted'].str.contains(u'该内容已被发布者删除') ] 
# data.cutted[:5]

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/sv/2x1mwq693cn_rg15vfcm7m6c0000gn/T/jieba.cache
Loading model cost 0.788 seconds.
Prefix dict has been built succesfully.


In [43]:
# Save word_list
# f=open("word_list.txt","w")
# s=""
# for i in range(len(word_list)):
#     s=s+word_list[i]+", "
#     if i>0 and i%15.0==0: 
#         print(s,file=f)
#         s=""
# f.close()

In [ ]:
# Load corpus and hit by word_list
# Only first time, after that could use small corpus after hit.

# wordvec=pd.read_csv("/Users/RyanQu/Downloads/sgns.merge.word", delimiter=" ", skiprows=1,header=None,engine="c",quoting=csv.QUOTE_NONE, encoding='utf-8')
# wordvec = wordvec.iloc[:, :-1] # delete last row, empty row
# wordvec_list=wordvec.iloc[:, 0].tolist() # pick the first row, the words
# Generate small_wordvec
# small_wordvec = pd.DataFrame(data=None, columns=wordvec.columns)
# for word in word_list:
#     if word in wordvec_list: 
#         i=wordvec_list.index(word)
#         small_wordvec=small_wordvec.append(wordvec.iloc[i],ignore_index=True)
# 
# small_wordvec.to_csv("TopicModel_Output/Small_wordvec.csv")
small_wordvec = pd.read_csv("TitleModel_Output/Small_wordvec.csv",header=None)

In [7]:
## Calculate avg vector for title (long time processing)

small_wordvec_list=small_wordvec.iloc[:, 1].tolist()
average_doc_vec = pd.DataFrame(data=None, columns=small_wordvec.columns)
for index, row in data.iterrows():
    temp=[0]*300
    temp=np.array(temp)
    count=1
    temp_word_list = row["cutted"].split(" ")
    for word in temp_word_list:
        if word in small_wordvec_list:
            i=small_wordvec_list.index(word)
            temp=temp+small_wordvec.iloc[i,2:].values
            count=count+1
    temp=temp/float(count)
    temp=np.append([index],temp)
    temp=pd.Series(temp)
    average_doc_vec=average_doc_vec.append(temp,ignore_index=True)
    
average_doc_vec.to_csv("TitleModel_Output/word2vec_average_model.csv")

In [8]:
## Cover count

# temp=[]
# temp_wordsonly=[]
# temp_notin=[]
# for i in word_list:
#     if i in small_wordvec_list: 
#         temp.append([i,small_wordvec_list.index(i)])
#         temp_wordsonly.append(i)
#     else: temp_notin.append(i)
        
# len(temp) 1678/2040 words hit
# len(temp_notin) 362/2040 words not hit

# count_atleast1=0
# count_overhalf=0
# count_only1=0
# count_only1_list=[]
# count_overhalf_list=[]
# for row in cut_list_re_str:
#     l=row.split(" ")
#     flag=0.0
#     for i in l:
#         if i in temp_wordsonly:
#             flag+=1
#     if flag>0: count_atleast1+=1
#     if flag/len(l)>=0.5: 
#         count_overhalf+=1
#         count_overhalf_list.append(row)
#     if flag==1: 
#         count_only1+=1
#         count_only1_list.append(row)
# print(count_atleast1,count_overhalf,count_only1,len(cut_list_re_str))
# 12797 12752 337 12832 

# f=open("cover_only_one.txt","w")
# s=""
# for i in range(len(count_only1_list)):
#     s=s+count_only1_list[i]+", "
#     if i>0 and i%15.0==0: 
#         print(s,file=f)
#         s=""
# f.close()

In [4]:
data=data[~data.Title.isin(red_pocket_title)]

In [5]:
n_features = 1000
tf_vectorizer = CountVectorizer(strip_accents = 'unicode',
                                max_features=n_features,
                                max_df = 0.5,
                                min_df = 10)
tf = tf_vectorizer.fit_transform(data.cutted)
n_topics = 3
lda = LatentDirichletAllocation(n_components=n_topics, max_iter=50,
                                learning_method='online',
                                learning_offset=50.,
                                random_state=0)

lda.fit(tf)

LatentDirichletAllocation(batch_size=128, doc_topic_prior=None,
             evaluate_every=-1, learning_decay=0.7,
             learning_method='online', learning_offset=50.0,
             max_doc_update_iter=100, max_iter=50, mean_change_tol=0.001,
             n_components=3, n_jobs=None, n_topics=None, perp_tol=0.1,
             random_state=0, topic_word_prior=None,
             total_samples=1000000.0, verbose=0)

In [6]:
f=open("TitleModel_Output/stat.txt","a")
print("Features_%d, Topics_%d: " %(n_features,n_topics),file=f)
print("Perplexity, Score(log-likelihood): %f, %f " %(lda.perplexity(tf),lda.score(tf)),file=f)

topic_vec=lda.transform(tf)
topic_vec=pd.DataFrame(topic_vec[:,:])
column=["Topic_"+str(i) for i in range(n_topics)]
topic_vec.columns=column
topic_vec['Title']=list(data.Title)
topic_vec['Author']=list(data.Author)
topic_vec['Read']=list(data.Read)
topic_vec['Max_layer']=list(data.Max_layer)
topic_vec[["Read"]] = topic_vec[["Read"]].astype(int)
topic_vec['Time']=list(data.Time)
topic_vec['Stay']=list(data.Stay)
topic_vec['M_ratio']=list(data.M_ratio)

RP = []
topic = []
gap = []
for index, row in topic_vec.iterrows():
    if row.Title in red_pocket_title: RP.append(1)
    else: RP.append(0)
    _topic = row[column].tolist()
    if _topic==[_topic[0]]*len(_topic): topic.append('N/A')
    else: topic.append(_topic.index(max(_topic)))
        
topic_vec['RP']=RP
topic_vec['Topic']=topic

data_1 = pd.read_csv("mingjiang/mingjiang_total.csv",header=0)
data_1 = data_1[["Author","Dep","Job"]]
data_1 = data_1.drop_duplicates()
topic_vec = pd.merge(topic_vec, data_1, on='Author')

data_1 = pd.read_csv("LDA_His.csv",header=0)
data_1 = data_1[["Author","Title","sim_title","e_title"]]
topic_vec = pd.merge(topic_vec, data_1, on=['Author','Title'])

author_name = []
_author_fre = []
flag_list = []
flag_title_list = []
flag_author_list = []
grouped = data.groupby("Author")
for name, group in grouped:
    t1 = datetime.strptime(min(group["Time"])[:10], '%Y-%m-%d').date()
    t2 = datetime.strptime(max(group["Time"])[:10], '%Y-%m-%d').date()
    author_name.append(name)
    try:
        _author_fre.append(len(group)/(t2-t1).days)
    except:
        _author_fre.append(len(group)/1)
    _group = group.sort_values('Time')
    flag_author_list = flag_author_list + [name]*len(_group)
    flag_title_list = flag_title_list + _group.Title.tolist()
    flag_list = flag_list + [0]*math.floor(0.8*len(_group))+[1]*math.ceil(0.2*len(_group))

author_fre = pd.DataFrame()
author_fre["Author"] = author_name
author_fre["fre"] = _author_fre
topic_vec = pd.merge(topic_vec, author_fre, on='Author')

flag_df = pd.DataFrame()
flag_df["Author"] = flag_author_list
flag_df["Title"] = flag_title_list
flag_df["Flag"] = flag_list
topic_vec = pd.merge(topic_vec, flag_df, on=['Author','Title'])

topic_vec=topic_vec[topic_vec.Read>1]

topic_vec.to_csv("TitleModel_Output/Stat_Title_model_topic_vec_features_%d_topics_%d.csv" %(n_features,n_topics))

def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean([(np.abs(y_true[i] - y_pred[i]) / max(y_true[i],0.1)) for i in range(len(y_true))])
#     return np.mean(np.abs((y_true - y_pred) / y_true+0.01)) * 100


y=np.array([math.log(int(i)) for i in list(topic_vec["Read"])])
X=topic_vec[column]
reg = LinearRegression().fit(X, y)
y_pred=reg.predict(X)
mse=mean_squared_error(y, y_pred)
mape=mean_absolute_percentage_error(y, y_pred)
r2=r2_score(y, y_pred)
msle=mean_squared_log_error(y, y_pred)
print("Prediction on Log of Read: ", file=f)
print("MSE, MSLE, R^2, MAPE: %.3f, %.3f, %.5f, %.2f%% " %(mse,msle,r2,mape),file=f)
y=np.array([int(i) for i in list(topic_vec["Max_layer"])])
reg = LinearRegression().fit(X, y)
y_pred=reg.predict(X)
mse=mean_squared_error(y, y_pred)
mape=mean_absolute_percentage_error(y, y_pred)
r2=r2_score(y, y_pred)
msle=mean_squared_log_error(y, y_pred)
print("Prediction on Max Layers: ", file=f)
print("MSE, MSLE, R^2, MAPE: %.3f, %.3f, %.5f, %.2f%% " %(mse,msle,r2,mape),file=f)

def print_top_words(model, feature_names, n_top_words):
    s=""
    for topic_idx, topic in enumerate(model.components_):
        s+="Topic #%d:" % topic_idx
        s+=(" ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]))
        s+="\n"
    print(s)
    print(s,file=f)

n_top_words = 20
tf_feature_names = tf_vectorizer.get_feature_names()
print_top_words(lda, tf_feature_names, n_top_words)
f.close()

Topic #0:设计 特权 转发 装修 红包 周年 店庆 盛大 匠心 惊喜 大赛 年度 启动 邀请函 开有 装饰 全城 疯狂 手要 眼要
Topic #1:装修 设计 知识 家装 地方 省钱 细节 客厅 设计师 家居 风格 预算 公司 注意事项 千万 风水 房子 完美 合适 卫生间
Topic #2:装饰 感恩 工地 工艺 全景 风格 中国 答谢会 家里 温暖 全城 家装 万元 生活 巨款 下雨天 大德 惠来 品质 雅居



In [55]:
pyLDAvis.enable_notebook()
pyLDAvis.sklearn.prepare(lda, tf, tf_vectorizer)

In [26]:
# author_list=[u"薛剑锋",u"吴灿",u"江蓉",u"李玉强",u"蒋小村",u"曾小纾"]
c=Counter(data.iloc[:, 1].tolist())
# author_list, author_count=list(c.keys()), list(c.values())
for key, value in c.items():
    # 1067 in total, 249 >10, 129 >20, 81 >30
    if value>30:
        data_1=topic_vec[topic_vec.Author==key]
        plt.figure()
        plt.ylim(0,1)
        for index, row in data_1.iterrows():
            x=float(row.TopicRP)
            y=float(row.TopicNRP)
            l=row.Read
            if x>0.5: color='mediumblue'
            elif x==0.5: color='cornflowerblue'
            else: color='burlywood'
            plt.xlabel(u'属于刺激物欲类文章的概率')
            plt.ylabel(u'属于其他文章的概率')
            plt.text(0.75, 0.95, "圆心: 概率坐标", size = 8)
            plt.text(0.75, 0.87, "大小: 总阅读数", size = 8)
            plt.text(0.75, 0.79, "颜色: >,<,=0.5", size = 8)
            plt.title(key)
            plt.scatter(x,y,s=l,alpha=0.5,c=color)
        plt.savefig("Radar_Graph/%s_Radar.png" %key, format="PNG",dpi=600)
        plt.clf()

In [89]:
data_2=list(set(data.iloc[:, 1].tolist()))
data_2

In [27]:
key=u"李玉强"
data_1=topic_vec[topic_vec.Author==key]
plt.figure()
plt.ylim(0,1)
for index, row in data_1.iterrows():
    x=float(row.TopicRP)
    y=float(row.TopicNRP)
    l=row.Read
    if x>0.5: color='mediumblue'
    elif x==0.5: color='cornflowerblue'
    else: color='burlywood'
    plt.xlabel(u'属于刺激物欲类文章的概率')
    plt.ylabel(u'属于其他文章的概率')
    plt.text(0.75, 0.95, "圆心: 概率坐标", size = 8)
    plt.text(0.75, 0.87, "大小: 总阅读数", size = 8)
    plt.text(0.75, 0.79, "颜色: >,<,=0.5", size = 8)
    plt.title(key)
    plt.scatter(x,y,s=l,alpha=0.5,c=color)
plt.savefig("Radar_Graph/%s_Radar.png" %key, format="PNG",dpi=600)
plt.clf()